# Scored Memory Retriever
> Retrieve persistent memory entries by embedding similarity with score decay.

| Module | `anchor.retrieval` |
|--------|--------------------|
| Key classes | `ScoredMemoryRetriever`, `MemoryRetrieverAdapter`, `InMemoryEntryStore` |
| Difficulty | Intermediate |

`ScoredMemoryRetriever` searches a persistent entry store using embeddings.
`MemoryRetrieverAdapter` wraps it to conform to the standard `Retriever` protocol.

**Time:** ~7 minutes

## Setup

In [ ]:
from anchor.retrieval import ScoredMemoryRetriever, MemoryRetrieverAdapter
from anchor.storage import InMemoryEntryStore
from anchor.models import MemoryEntry, QueryBundle
from datetime import datetime, timedelta, timezone

## Populate the Entry Store
Create memory entries with varying ages.

In [ ]:
entry_store = InMemoryEntryStore()
now = datetime.now(timezone.utc)

entries = [
    MemoryEntry(entry_id="mem-1", content="User prefers Python for scripting tasks.",
               created_at=now - timedelta(hours=1)),
    MemoryEntry(entry_id="mem-2", content="User is working on a Rust side project.",
               created_at=now - timedelta(hours=24)),
    MemoryEntry(entry_id="mem-3", content="User asked about Go concurrency patterns.",
               created_at=now - timedelta(hours=72)),
    MemoryEntry(entry_id="mem-4", content="User needs help with Python data pipelines.",
               created_at=now - timedelta(hours=2)),
    MemoryEntry(entry_id="mem-5", content="User mentioned interest in TypeScript migration.",
               created_at=now - timedelta(hours=168)),
]

for entry in entries:
    entry_store.put(entry)

print(f"Entry store populated with {len(entries)} memories.")

## Create the Scored Memory Retriever

In [ ]:
def embed_fn(text: str) -> list[float]:
    padded = text[:128].ljust(128)
    return [hash(c) % 100 / 100.0 for c in padded]

scored = ScoredMemoryRetriever(
    store=entry_store,
    embed_fn=embed_fn,
)

print(f"Retriever type: {type(scored).__name__}")

## Retrieve Memories by Query

In [ ]:
query = QueryBundle(query_str="Python scripting")
results = scored.retrieve(query, top_k=3)

print(f"Query: '{query.query_str}'")
print(f"Results ({len(results)}):\n")
for i, item in enumerate(results):
    print(f"  [{i+1}] score={item.score:.4f}  {item.content[:60]}")

## Wrap with MemoryRetrieverAdapter
The adapter makes `ScoredMemoryRetriever` conform to the standard `Retriever`
protocol, allowing it to be used in pipelines alongside dense/sparse retrievers.

In [ ]:
adapter = MemoryRetrieverAdapter(retriever=scored)

# The adapter uses the same retrieve interface
results = adapter.retrieve(QueryBundle(query_str="Rust project"), top_k=2)

print(f"Adapter type: {type(adapter).__name__}")
print(f"Results via adapter ({len(results)}):\n")
for i, item in enumerate(results):
    print(f"  [{i+1}] score={item.score:.4f}  {item.content[:60]}")

## Key Takeaways
- `ScoredMemoryRetriever` searches persistent memory entries by embedding similarity
- Entries with recency and relevance combine for final scores
- `MemoryRetrieverAdapter` wraps it to fit the standard `Retriever` protocol
- Use the adapter to include memory retrieval in hybrid pipelines

**Next:** [Async Retrievers](05_async_retrievers.ipynb)